# 00 — Setup & test de bout en bout

Ce notebook vérifie que tous les modules s'importent correctement et lance un premier
pricing avec des **paramètres de marché provisoires** (à remplacer une fois les données
réelles calibrées dans `data_loader.py` / `market_calibration.py`).

TODO : remplacer les valeurs de `spot`, `risk_free_rate`, `dividend_yield`, `volatility`
ci-dessous par les valeurs calibrées depuis les données de marché réelles.

In [8]:
import sys
sys.path.append("../src")

import numpy as np

from product.product_specs import ProductSpecs
from simulation.gbm_simulator import GBMParameters
from pricing.monte_carlo_engine import MonteCarloEngine
from pricing.convergence import run_convergence_study
from risk.greeks import compute_greeks
from risk.stress_testing import run_stress_tests, DEFAULT_SCENARIOS

## 1. Chargement des spécifications du produit (depuis la config YAML)

In [9]:
specs = ProductSpecs.from_config("../config/product_config.yaml")
specs.validate()
print(f"Nombre d'observations : {specs.n_observations}")
print(f"Barrières d'autocall  : {specs.autocall_barriers}")
print(f"Barrières de coupon   : {specs.coupon_barriers}")
print(f"Coupon par période    : {specs.coupon_rate_period:.4%}")
print(f"Knock-in              : {specs.knock_in_barrier:.0%} ({specs.knock_in_observation})")

Nombre d'observations : 12
Barrières d'autocall  : [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
Barrières de coupon   : [0.7 0.7 0.7 0.7 0.7 0.7 0.7 0.7 0.7 0.7 0.7 0.7]
Coupon par période    : 2.2500%
Knock-in              : 60% (at_maturity)


## 2. Paramètres de marché — PROVISOIRES

TODO : remplacer par calibration réelle (yfinance + FRED + VIX/VSTOXX).

In [10]:
# --- Valeurs provisoires, à remplacer une par une ---
market_params = GBMParameters(
    spot=100.0,             # TODO: dernier prix réel du sous-jacent
    risk_free_rate=0.03,    # TODO: taux calibré depuis FRED
    dividend_yield=0.02,    # TODO: dividend yield réel de l'indice
    volatility=0.20,        # TODO: vol calibrée (VIX/VSTOXX ou historique)
)
market_params

GBMParameters(spot=100.0, risk_free_rate=0.03, dividend_yield=0.02, volatility=0.2, reference_spot=None)

## 3. Pricing de référence

In [11]:
engine = MonteCarloEngine(specs, market_params)
result = engine.price(n_paths=100_000, random_seed=42, antithetic=True)

print(f"Prix estimé          : {result.price:.2f}")
print(f"Erreur standard      : {result.std_error:.4f}")
print(f"IC 95%               : [{result.confidence_interval_95[0]:.2f}, {result.confidence_interval_95[1]:.2f}]")
print(f"Proba knock-in       : {result.knock_in_probability:.2%}")
print(f"Proba autocall/date  : {np.round(result.autocall_probability_by_date, 4)}")

Prix estimé          : 1024.16
Erreur standard      : 0.3573
IC 95%               : [1023.46, 1024.86]
Proba knock-in       : 8.20%
Proba autocall/date  : [0.4899 0.1227 0.0617 0.0383 0.0263 0.0214 0.0163 0.0128 0.011  0.0092
 0.008  0.0064]


## 4. Étude de convergence

In [12]:
conv = run_convergence_study(engine, path_counts=[1000, 5000, 10000, 50000, 100000, 200000])
for n, p, se in zip(conv.path_counts, conv.prices, conv.std_errors):
    print(f"n={n:>7} | prix={p:8.2f} | std_error={se:.4f}")

n=   1000 | prix= 1018.48 | std_error=3.7615
n=   5000 | prix= 1020.22 | std_error=1.6797
n=  10000 | prix= 1021.84 | std_error=1.1652
n=  50000 | prix= 1024.11 | std_error=0.5049
n= 100000 | prix= 1024.16 | std_error=0.3573
n= 200000 | prix= 1024.17 | std_error=0.2531


## 5. Greeks

In [13]:
greeks = compute_greeks(specs, market_params, n_paths=100_000)
print(f"Prix de base : {greeks.base_price:.2f}")
print(f"Delta        : {greeks.delta:.4f}")
print(f"Gamma        : {greeks.gamma:.6f}")
print(f"Vega         : {greeks.vega:.4f}")
print(f"Theta        : {greeks.theta:.4f}")
print(f"Rho          : {greeks.rho:.4f}")

Prix de base : 1024.16
Delta        : -0.3872
Gamma        : -0.214336
Vega         : -429.6758
Theta        : 48.7479
Rho          : -684.8028


## 6. Stress tests

In [14]:
stress_results = run_stress_tests(specs, market_params, n_paths=100_000)
for r in stress_results:
    print(f"{r.scenario_name:<40} | prix={r.price:8.2f} | Δ={r.price_change_pct:+6.2f}% | "
          f"proba KI={r.knock_in_probability:6.2%} (Δ={r.knock_in_probability_change:+.2%})")

Choc de volatilité modéré (+10 vol pts)  | prix=  980.35 | Δ= -4.28% | proba KI=21.59% (Δ=+13.39%)
Choc de volatilité sévère (+25 vol pts)  | prix=  920.43 | Δ=-10.13% | proba KI=38.03% (Δ=+29.83%)
Crash type 2008                          | prix=  628.07 | Δ=-38.67% | proba KI=65.56% (Δ=+57.36%)
Crash type mars 2020                     | prix=  696.63 | Δ=-31.98% | proba KI=61.11% (Δ=+52.91%)
Hausse des taux (+100bp)                 | prix= 1017.04 | Δ= -0.70% | proba KI= 7.00% (Δ=-1.20%)
Baisse des taux (-100bp)                 | prix= 1031.19 | Δ= +0.69% | proba KI= 9.63% (Δ=+1.44%)
